# Lab 09: Stationarity, ACF/PACF และ ARIMA

รายวิชา `229351: Statistical Learning for Data Science 1`

Lab นี้ใช้อนุกรม `CarSales`, `CM_temp` และ `AirPassengers` ชุดเดียวกับที่ใช้ในบทเรียน Lec 09 ตัวเลขทุกตัวที่ได้จึงตรงกับตัวเลขบนสไลด์ ถ้าค่าที่ได้ไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด ไม่ใช่เรื่องของการปัดเศษ

Lab นี้เขียน `r_k` และ grid search ด้วยมือก่อน แล้วจึงใช้ `statsmodels` เป็นตัวตรวจคำตอบ ไม่ใช่ในทางกลับกัน

## เป้าหมาย

1. ทดสอบ stationarity ด้วย KPSS และ ADF โดยอ่านทิศทางของสมมติฐานทั้งสองได้ถูกต้อง
2. อธิบายได้ว่าเหตุใด `p = 0.0100` จาก KPSS จึงไม่ใช่ตัวเลขที่วัดได้ และเหตุใด warning จึงสำคัญ
3. เขียน `r_k` ด้วยมือตามสูตรบนสไลด์ และตรวจกับ `statsmodels.tsa.stattools.acf`
4. อ่าน trend และ seasonality ออกจากรูปร่างของ ACF ได้
5. เลือก `D` จากกราฟและ ACF และเลือก `d` จากการทดสอบ ตามลำดับที่ถูกต้อง
6. ระบุ over-differencing จาก sd ที่เพิ่มขึ้นและ `r_1` ที่ติดลบมาก
7. อ่านอันดับ `p` และ `q` เบื้องต้นจาก ACF/PACF ของอนุกรมที่ difference แล้ว
8. เขียน grid search บน AICc ด้วยตนเอง แทน `auto_arima` ที่ติดตั้งไม่ได้
9. อธิบายและวัดได้ว่าเหตุใด AICc จึงเปรียบเทียบข้าม `d` ไม่ได้
10. ตรวจ residual ด้วย Ljung-Box โดยอ่านทิศทางของสมมติฐานได้ถูก
11. เปรียบเทียบ ETS กับ ARIMA บน test set เดียวกับ Lab 08 และรายงานผลอย่างซื่อสัตย์

## 0. Setup

In [ ]:
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.stats.diagnostic import acorr_ljungbox

pd.set_option("display.float_format", lambda value: f"{value:.6g}")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

# การ fit หลายร้อยครั้งใน lab นี้จะพ่น ConvergenceWarning ออกมาเป็นระยะ ปิดเสียงไว้
# ตรงนี้ครั้งเดียว ส่วน Exercise 1 จะเปิด warning กลับมาเฉพาะจุดเพื่อให้เห็น
# InterpolationWarning ของ KPSS ซึ่งเป็น warning ที่ต้องอ่าน ไม่ใช่ที่ต้องปิด
warnings.simplefilter("ignore")

# ข้อมูลถูก pin ไว้ที่ commit หนึ่งโดยเฉพาะ เพื่อให้คำตอบเชิงตัวเลขไม่เปลี่ยนตามเวลา
PIN = "8d2d9c8ebfd86113c62a1f994f88e6e6732b0277"
BASE = f"https://raw.githubusercontent.com/skitimo/ds351-data/{PIN}"
print(BASE)

## 1. ข้อมูลและดัชนีเวลา (8 คะแนน)

Lab นี้ใช้สามอนุกรม

| ไฟล์ | เนื้อหา | ความถี่ดิบ |
|---|---|---|
| `CarSales.csv` | ยอดขายรถยนต์ในควิเบก 1960-01 ถึง 1968-12 | รายเดือน 108 แถว |
| `CM_temp.csv` | อุณหภูมิสูงสุดในเชียงใหม่ 1998-01-01 ถึง 2019-07-31 | **รายวัน** 7882 แถว |
| `AirPassengers.csv` | จำนวนผู้โดยสารรายเดือน 1949-01 ถึง 1960-12 | รายเดือน 144 แถว |

**สามกับดักเดิมจาก Lab 08 ยังอยู่**

| เขียนแบบนี้ | ผลลัพธ์ |
|---|---|
| `resample("M")` | `ValueError` ใน pandas 3 ให้ใช้ `"ME"` หรือ `"MS"` |
| `read_csv(..., date_parser=...)` | `TypeError` ถูกถอดออกตั้งแต่ pandas 2.0 |
| ไม่ตั้ง frequency ให้ index | `SARIMAX` จะตีความ `seasonal_order` ผิด ให้เรียก `.asfreq(...)` ทุกครั้ง |

การแบ่ง train/test ใช้ธรรมเนียมเดียวกับ Lab 07 และ Lab 08 คือ `y[:-k]` และ `y[-k:]` โดย `k = 12` สำหรับ `CarSales` และ `k = 24` สำหรับ `CM_temp` **ช่วงทดสอบของ `CM_temp` เป็นช่วงเดียวกับ Lab 08 เป๊ะ ๆ** ซึ่งเป็นสิ่งที่ทำให้ Exercise 6 เปรียบเทียบได้

In [ ]:
raw_sales = pd.read_csv(f"{BASE}/CarSales.csv")
raw_temp = pd.read_csv(f"{BASE}/CM_temp.csv", parse_dates=["Date"])
raw_air = pd.read_csv(f"{BASE}/AirPassengers.csv")
print(len(raw_sales), len(raw_temp), len(raw_air))
display(raw_sales.head(3))

In [ ]:
# TODO: สร้างสามอนุกรมที่มี DatetimeIndex พร้อม frequency
#       cars     : CarSales รายเดือน  (คอลัมน์ Month, Sales) freq "MS"
#       temp_m   : CM_temp เฉลี่ยรายเดือน                    freq "MS"
#       air      : AirPassengers รายเดือน (Month, Passengers) freq "MS"
#       คำแนะนำ: คอลัมน์ Month ของ CarSales และ AirPassengers เป็น string "YYYY-MM"
#                ส่วน CM_temp เป็นรายวัน ต้อง resample ก่อน
cars = ...
temp_m = ...
air = ...

for name, s in (("cars", cars), ("temp_m", temp_m), ("air", air)):
    print(f"{name:8s} n={len(s):4d}  {s.index[0]:%Y-%m} .. {s.index[-1]:%Y-%m}  freq={s.index.freqstr}")

In [ ]:
assert len(cars) == 108 and cars.index.freqstr == "MS"
assert len(temp_m) == 259 and temp_m.index.freqstr == "MS"
assert len(air) == 144 and air.index.freqstr == "MS"
assert f"{temp_m.index[-1]:%Y-%m}" == "2019-07"

cars_train, cars_test = cars[:-12], cars[-12:]
temp_train, temp_test = temp_m[:-24], temp_m[-24:]
assert (len(cars_train), len(cars_test)) == (96, 12)
assert (len(temp_train), len(temp_test)) == (235, 24)
assert f"{temp_test.index[0]:%Y-%m}" == "2017-08"   # the Lab 08 window, exactly
print("ok: three series, two splits")

In [ ]:
def rmse(actual, predicted):
    """RMSE, the same metric Lab 07 and Lab 08 used."""
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(predicted)) ** 2)))


def seasonal_naive(train, index, m=12):
    """The benchmark with no fitted parameters: last year's same month."""
    return pd.Series([train.iloc[-m + (i % m)] for i in range(len(index))], index=index)


print(f"{rmse(cars_test, seasonal_naive(cars_train, cars_test.index)):.1f}")

## Exercise 1: KPSS และ ADF (14 คะแนน)

การทดสอบสองตัวนี้มีสมมติฐานหลัก **กลับด้านกัน**

| | $H_0$ | ปฏิเสธ $H_0$ แปลว่า |
|---|---|---|
| KPSS | อนุกรมเป็น stationary | **ต้อง** difference |
| ADF | อนุกรมมี unit root | **ไม่ต้อง** difference |

เขียนฟังก์ชันเดียวที่รันทั้งสองตัวและแปลผลให้ถูกทิศทาง แล้วใช้กับ `cars` ทั้งสี่รูปแบบ

In [ ]:
# TODO: เขียน run_tests(series) ให้คืน dict ที่มีคีย์
#       kpss_stat, kpss_p, adf_stat, adf_p, kpss_says, adf_says
#       โดย kpss_says และ adf_says เป็น "stationary" หรือ "not stationary"
#       ระวังทิศทาง: KPSS ปฏิเสธ H0 -> not stationary, ADF ปฏิเสธ H0 -> stationary
#       คำแนะนำ: kpss(series, nlags="auto") คืน (stat, p, lags, crit)
#                adfuller(series) คืน tuple ที่สองค่าแรกคือ stat กับ p
def run_tests(series):
    ...


rows = {}
for label, s in (("level", cars),
                 ("d=1", cars.diff().dropna()),
                 ("D=1", cars.diff(12).dropna()),
                 ("d=1,D=1", cars.diff().diff(12).dropna())):
    rows[label] = {"n": len(s), "sd": float(s.std()), **run_tests(s)}
table1 = pd.DataFrame(rows).T
display(table1[["n", "sd", "kpss_stat", "kpss_p", "adf_stat", "adf_p", "kpss_says", "adf_says"]])

In [ ]:
# ตรงกับตารางบนสไลด์ Lec 09 หน้า "ผลลัพธ์บน CarSales"
assert abs(rows["level"]["kpss_stat"] - 1.2501) < 5e-4
assert abs(rows["level"]["adf_stat"] + 1.2238) < 5e-4
assert abs(rows["level"]["adf_p"] - 0.6633) < 5e-4
assert abs(rows["D=1"]["kpss_stat"] - 0.1630) < 5e-4
assert abs(rows["D=1"]["adf_p"] - 0.0050) < 5e-4
assert abs(rows["D=1"]["sd"] - 1766.6) < 0.1
assert rows["level"]["kpss_says"] == rows["level"]["adf_says"] == "not stationary"
assert rows["D=1"]["kpss_says"] == rows["D=1"]["adf_says"] == "stationary"
print("ok: Exercise 1 matches the slides")

### กับดัก: `p` ที่ถูกตัดที่ขอบตาราง

`statsmodels` เปิดตารางค่าวิกฤตที่มีจำกัด แล้วตัดค่า `p` ไว้ที่ขอบตาราง พร้อมส่ง `InterpolationWarning` ออกมา ซึ่งเป็น **warning ไม่ใช่ error** เซลล์ถัดไปจับ warning นั้นมาแสดงให้เห็น

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    kpss(cars, nlags="auto")
    kpss(cars.diff(12).dropna(), nlags="auto")
for w in caught:
    print(type(w.message).__name__, "->", str(w.message).split(".")[0].strip())

**เขียนคำตอบข้อ 1** (ตอบให้ครบทั้งสามข้อ)

1. `cars` ที่ระดับ ได้ KPSS `p = 0.0100` และ ADF `p = 0.6633` การทดสอบสองตัวนี้เห็นตรงกันหรือขัดกัน และสรุปว่าอย่างไร
2. `p = 0.0100` ที่ KPSS รายงานมา แปลว่าอะไรกันแน่ และเหตุใดจึงไม่ควรนำไปเทียบกับ `p = 0.0100` ของอนุกรมอื่นว่า "มีนัยสำคัญเท่ากัน"
3. ถ้ามีคนสรุปว่า "ADF ให้ `p = 0.6633` ซึ่งมากกว่า 0.05 แปลว่าอนุกรมนี้ stationary" เขาอ่านผิดตรงไหน

_(เขียนคำตอบที่นี่)_

## Exercise 2: `r_k` ด้วยมือ (16 คะแนน)

สูตรบนสไลด์คือ

$$ r_k \;=\; \frac{\sum_{t=k+1}^{T} (y_t - \bar y)(y_{t-k} - \bar y)}{\sum_{t=1}^{T} (y_t - \bar y)^2} $$

สังเกตว่าตัวเศษเสีย $k$ พจน์ไป แต่ตัวส่วนยังรวมครบ $T$ พจน์ เขียนให้ตรงตามนี้ แล้วตรวจกับ `statsmodels`

In [ ]:
# TODO: เขียน acf_by_hand(y, nlags) ให้คืน array ความยาว nlags+1
#       โดย index 0 คือ r_0 = 1.0 และ index k คือ r_k ตามสูตรข้างบน
#       ห้ามเรียก acf() ในฟังก์ชันนี้
#       คำแนะนำ: ถ้า dev = y - y.mean() แล้ว ตัวเศษของ r_k คือ dev[k:] @ dev[:-k]
def acf_by_hand(y, nlags):
    ...


mine = acf_by_hand(cars_train, 24)
theirs = acf(cars_train.to_numpy(), nlags=24, fft=False)
gap = float(np.max(np.abs(mine - theirs)))
print(f"max |mine - statsmodels| over lags 0..24 = {gap:.3e}")

In [ ]:
assert gap < 1e-10, "the hand-written r_k does not match statsmodels"
assert abs(mine[0] - 1.0) < 1e-12
print("ok: r_k written from the slide formula reproduces statsmodels exactly")

### แถบนัยสำคัญ และรูปร่างของ ACF

ถ้าอนุกรมเป็น white noise จริง $r_k$ จะกระจายรอบศูนย์ด้วย sd ประมาณ $1/\sqrt{T}$ แถบ $\pm 1.96/\sqrt{T}$ จึงเป็นช่วงที่ $r_k$ ควรอยู่ข้างใน

ต่อไปใช้ `acf_by_hand` สร้างตัวเลขเดียวกับสไลด์หน้า "trend กับ seasonality หน้าตาต่างกันใน ACF" ซึ่งเทียบ **อนุกรมที่มี trend** กับ **อนุกรมที่ไม่มี trend**

In [ ]:
def band(n):
    return 1.96 / np.sqrt(n)


shape = pd.DataFrame(
    {name: {f"r_{k}": acf_by_hand(s, 36)[k] for k in (1, 2, 3, 6, 12, 24, 36)}
     for name, s in (("AirPassengers", air), ("CM_temp", temp_m))}
).T
shape["band"] = [band(len(air)), band(len(temp_m))]
display(shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), sharey=True)
for ax, (name, s) in zip(axes, (("AirPassengers", air), ("CM_temp", temp_m))):
    r = acf_by_hand(s, 36)
    ax.bar(range(1, 37), r[1:], width=0.6)
    ax.axhline(band(len(s)), ls="--", color="crimson")
    ax.axhline(-band(len(s)), ls="--", color="crimson")
    ax.set_title(name)
    ax.set_xlabel("Lag k")
plt.tight_layout()
plt.show()

In [ ]:
# ตรงกับตารางบนสไลด์ Lec 09 หน้า "trend กับ seasonality หน้าตาต่างกันใน ACF"
assert abs(shape.loc["AirPassengers", "r_1"] - 0.948) < 5e-4
assert abs(shape.loc["AirPassengers", "r_12"] - 0.760) < 5e-4
assert abs(shape.loc["AirPassengers", "r_36"] - 0.337) < 5e-4
assert abs(shape.loc["CM_temp", "r_6"] + 0.126) < 5e-4
assert abs(shape.loc["CM_temp", "r_12"] - 0.736) < 5e-4
assert abs(shape.loc["CM_temp", "r_36"] - 0.695) < 5e-4
print("ok: Exercise 2 matches the slides")

**เขียนคำตอบข้อ 2** (ตอบให้ครบทั้งสามข้อ)

1. ทั้งสองอนุกรมมียอดแหลมที่ lag 12, 24, 36 เหมือนกัน แต่ของ AirPassengers **จางลง** ($0.760 \to 0.532 \to 0.337$) ขณะที่ของ CM_temp **ไม่จาง** ($0.736 \to 0.689 \to 0.695$) อธิบายว่าเพราะอะไร
2. CM_temp มี $r_6 = -0.126$ ซึ่งติดลบ ส่วน AirPassengers ไม่มี $r_k$ ที่ติดลบเลยใน 36 lag แรก อธิบายว่าเพราะอะไร
3. แถบนัยสำคัญของ CM_temp ($\pm 0.122$) แคบกว่าของ AirPassengers ($\pm 0.163$) เพราะอะไร และนั่นทำให้สรุปได้ไหมว่า CM_temp "มี autocorrelation มากกว่า"

_(เขียนคำตอบที่นี่)_

## Exercise 3: เลือก `D` และ `d` และรู้ว่าเมื่อไรเกิน (14 คะแนน)

ลำดับที่ถูกต้องคือ

1. เลือก `D` จาก **กราฟและ ACF** ไม่ใช่จากการทดสอบ
2. เลือก `d` จาก **KPSS** บนอนุกรมที่ seasonal difference แล้ว

ข้อ 1 สำคัญเพราะ KPSS และ ADF **มองไม่เห็น seasonality** เซลล์แรกของข้อนี้พิสูจน์ให้ดู

In [ ]:
# elecequip ไม่ได้อยู่ใน lab นี้ แต่ CM_temp ทำหน้าที่เดียวกันได้: seasonality แรงมาก
proof = pd.DataFrame({
    "temp_m level": run_tests(temp_m),
    "temp_m D=1": run_tests(temp_m.diff(12).dropna()),
}).T[["kpss_p", "kpss_says", "adf_p", "adf_says"]]
display(proof)
print("r_12 ของ temp_m ที่ยังไม่ difference =", round(acf_by_hand(temp_m, 12)[12], 3))

In [ ]:
# TODO: สร้าง over ซึ่งเป็น DataFrame แสดงว่า ถ้า difference เพิ่มขึ้นเรื่อย ๆ
#       หลังจาก D=1 แล้ว sd และ r_1 เปลี่ยนไปอย่างไร
#       แถวคือจำนวน first difference ที่เพิ่ม (0, 1, 2, 3) คอลัมน์คือ n, sd, r_1
#       คำแนะนำ: เริ่มจาก base = cars.diff(12).dropna() แล้ววนลูป s = s.diff().dropna()
base = cars.diff(12).dropna()
records = []
s = base
for extra in range(4):
    ...
over = pd.DataFrame(records).set_index("extra_d")
display(over)

In [ ]:
# ตรงกับตารางบนสไลด์ Lec 09 หน้า "Over-differencing มีลายเซ็นของมันเอง"
assert abs(over.loc[0, "sd"] - 1766.6) < 0.1
assert abs(over.loc[1, "sd"] - 2222.2) < 0.1
assert abs(over.loc[2, "sd"] - 3919.9) < 0.1
assert abs(over.loc[3, "sd"] - 7308.7) < 0.1
assert abs(over.loc[0, "r_1"] - 0.217) < 5e-4
assert abs(over.loc[1, "r_1"] + 0.539) < 5e-4
assert over["sd"].idxmin() == 0, "the minimum sd should be at D=1 with no extra d"
print("ok: Exercise 3 matches the slides")

**เขียนคำตอบข้อ 3** (ตอบให้ครบทั้งสามข้อ)

1. จากตาราง `proof` `temp_m` ที่ยังไม่ difference ผ่านหรือไม่ผ่านการทดสอบทั้งสองตัว และ `r_12 = 0.736` บอกอะไรที่การทดสอบไม่ได้บอก สรุปเป็นกฎการทำงานหนึ่งประโยค
2. จากตาราง `over` ค่า `sd` ต่ำสุดอยู่ที่แถวไหน และ `r_1` เปลี่ยนจาก `0.217` เป็น `-0.539` เมื่อเพิ่ม `d` หนึ่งครั้ง อธิบายว่าเหตุใดการ difference จึงสร้างความสัมพันธ์เชิงลบขึ้นมาเอง
3. สรุปว่า `CarSales` ควรใช้ `d` และ `D` เท่าไร และเหตุใดคำตอบนี้จึงไม่ใช่ `d=1, D=1` ทั้งที่แถวนั้นก็ผ่านการทดสอบทุกตัวใน Exercise 1

_(เขียนคำตอบที่นี่)_

## Exercise 4: อ่านอันดับจาก ACF/PACF แล้ว fit (16 คะแนน)

กฎการอ่าน ใช้กับกราฟของอนุกรม **ที่ difference แล้ว** เท่านั้น

| รูปแบบ | แปลว่า |
|---|---|
| PACF ตัดหายหลัง lag $p$, ACF ค่อย ๆ ลด | ARIMA($p,d,0$) |
| ACF ตัดหายหลัง lag $q$, PACF ค่อย ๆ ลด | ARIMA($0,d,q$) |
| ทั้งคู่ค่อย ๆ ลด | $p, q > 0$ กราฟช่วยไม่ได้ |

ใช้ `cars_train` หลัง `D=1` ตามข้อสรุปของ Exercise 3

In [ ]:
# TODO: สร้าง diff_train = cars_train หลัง seasonal difference
#       แล้วคำนวณ r_acf กับ r_pacf ถึง lag 24 และแถบนัยสำคัญ b
#       ใช้ acf_by_hand สำหรับ ACF และ pacf(..., method="ywm") สำหรับ PACF
diff_train = ...
r_acf = ...
r_pacf = ...
b = ...

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for ax, vals, name in ((axes[0], r_acf, "ACF"), (axes[1], r_pacf, "PACF")):
    ax.bar(range(1, 25), vals[1:25], width=0.6)
    ax.axhline(b, ls="--", color="crimson")
    ax.axhline(-b, ls="--", color="crimson")
    ax.set_ylabel(name)
axes[1].set_xlabel("Lag k")
plt.tight_layout()
plt.show()

sig_acf = [k for k in range(1, 25) if abs(r_acf[k]) > b]
sig_pacf = [k for k in range(1, 25) if abs(r_pacf[k]) > b]
print(f"T = {len(diff_train)}, band = +/- {b:.3f}")
print("ACF  outside:", sig_acf)
print("PACF outside:", sig_pacf)

In [ ]:
# ตรงกับตารางบนสไลด์ Lec 09 หน้า "อ่านอันดับจาก CarSales จริง"
assert len(diff_train) == 84
assert abs(b - 0.214) < 5e-4
assert abs(r_acf[1] - 0.269) < 5e-4 and abs(r_acf[2] - 0.321) < 5e-4
assert abs(r_pacf[1] - 0.269) < 5e-4 and abs(r_pacf[2] - 0.267) < 5e-4
assert sig_acf == [1, 2, 12, 17]
assert sig_pacf == [1, 2, 12, 15, 17]
print("ok: Exercise 4 reading matches the slides")

In [ ]:
# TODO: fit ผู้สมัครที่อ่านได้จากกราฟ คือ SARIMA(0,0,2)(0,1,1,12)
#       ใช้ cars_train, trend="c" (ใช้ได้เพราะ d = 0) แล้วพยากรณ์ 12 เดือน
#       เก็บผลไว้ใน read_fit และ read_rmse
#       คำแนะนำ: SARIMAX(y, order=..., seasonal_order=..., trend="c",
#                        enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
#                อ่าน AICc ได้จาก .aicc และพยากรณ์ด้วย .forecast(12)
read_fit = ...
read_rmse = ...
sn_rmse = rmse(cars_test, seasonal_naive(cars_train, cars_test.index))
print(f"AICc = {read_fit.aicc:.2f}   test RMSE = {read_rmse:.1f}"
      f"   seasonal naive = {sn_rmse:.1f}")

In [ ]:
assert abs(read_fit.aicc - 1229.36) < 0.05
assert abs(read_rmse - 1655.73) < 0.5
assert abs(sn_rmse - 2290.83) < 0.5
print("ok: the model read off the plots is the one on the slides")

**เขียนคำตอบข้อ 4** (ตอบให้ครบทั้งสามข้อ)

1. ACF มีนัยสำคัญที่ lag 1 และ 2 แล้วเงียบ ส่วน PACF ก็มีนัยสำคัญที่ lag 1 และ 2 เช่นกัน ตามกฎในตารางข้างบน กรณีนี้อ่านได้ว่าอะไร และเหตุใดจึงยังเลือก `q = 2` เป็นผู้สมัครได้
2. ทั้ง ACF และ PACF หลุดออกนอกแถบที่ lag 12 ด้วย นั่นสื่อถึงพจน์แบบใด และเหตุใดจึงเป็น `Q = 1` ไม่ใช่ `q = 12`
3. lag 15 และ 17 ก็หลุดออกนอกแถบ ควรใส่พจน์ที่ lag 15 และ 17 เข้าไปด้วยหรือไม่ ให้เหตุผลเชิงตัวเลข

_(เขียนคำตอบที่นี่)_

## Exercise 5: grid search ที่เขียนเอง และกับดักของ `d` (16 คะแนน)

`pmdarima.auto_arima` **ติดตั้งไม่ได้** บน environment ของรายวิชา เพราะมันต้องการ numpy 1.x เราจึงเขียน grid search เอง ซึ่งดีกว่าในแง่การเรียนรู้อยู่แล้ว เพราะจะได้เห็นว่ากำลังเลือกอะไรอยู่

**กล่องที่จะค้น** `p, q` $\in \{0,1,2\}$ และ `P, Q` $\in \{0,1\}$ โดย `D = 1`, `m = 12` รวม 36 การ fit ต่อหนึ่งค่าของ `d`

สองสิ่งที่ต้องจัดการในลูป

- `SARIMAX` บาง order จะ **ขว้าง exception** ต้อง `try`/`except`
- บาง fit จะ **ไม่ลู่เข้า** แต่ยังคืน AICc ออกมา ต้องบันทึกไว้ใน `converged`

In [ ]:
# TODO: เขียน sarima_grid(train, test, d) ให้คืน DataFrame ที่มีคอลัมน์
#       order, seasonal, aicc, rmse, converged สำหรับทุก (p,q,P,Q) ในกล่อง
#       ใช้ trend="c" เมื่อ d == 0 และ trend=None เมื่อ d >= 1
#       คำแนะนำ: itertools.product(range(3), range(3), range(2), range(2))
#                res.mle_retvals.get("converged", True) บอกว่า fit ลู่เข้าหรือไม่
#                sort ตาม aicc ก่อน return
def sarima_grid(train, test, d, D=1, m=12):
    ...


grid_d0 = sarima_grid(cars_train, cars_test, d=0)
grid_d1 = sarima_grid(cars_train, cars_test, d=1)
print(f"d=0: {len(grid_d0)} fits, {(~grid_d0['converged']).sum()} did not converge")
print(f"d=1: {len(grid_d1)} fits, {(~grid_d1['converged']).sum()} did not converge")
display(grid_d0.head(3))

In [ ]:
best_d0 = grid_d0.iloc[0]
best_d1 = grid_d1.iloc[0]
compare = pd.DataFrame([
    {"branch": "d=0 (what the tests said)", "order": best_d0["order"],
     "seasonal": best_d0["seasonal"], "aicc": best_d0["aicc"], "rmse": best_d0["rmse"]},
    {"branch": "d=1 (the other branch)", "order": best_d1["order"],
     "seasonal": best_d1["seasonal"], "aicc": best_d1["aicc"], "rmse": best_d1["rmse"]},
]).set_index("branch")
display(compare)

cross_d = pd.concat([grid_d0, grid_d1]).sort_values("aicc").iloc[0]
print(f"\nถ้าปล่อยให้ d อยู่ในลูปด้วย AICc ต่ำสุดข้าม d จะเลือก:")
print(f"  {cross_d['order']}{cross_d['seasonal']}  AICc={cross_d['aicc']:.2f}"
      f"  test RMSE={cross_d['rmse']:.1f}")

In [ ]:
# ตรงกับตารางบนสไลด์ Lec 09 หน้า "ราคาของการเทียบข้าม d บนสามอนุกรม"
assert len(grid_d0) == 36 and len(grid_d1) == 36
assert abs(best_d0["aicc"] - 1229.36) < 0.05
assert abs(best_d0["rmse"] - 1655.73) < 0.5
assert abs(best_d1["aicc"] - 1209.40) < 0.05
assert abs(best_d1["rmse"] - 2266.97) < 0.5
assert best_d1["aicc"] < best_d0["aicc"], "the d=1 branch should have the lower AICc"
assert best_d1["rmse"] > best_d0["rmse"], "and the worse forecast"
assert best_d0["order"] == (0, 0, 2) and best_d0["seasonal"] == (0, 1, 1, 12)
print("ok: Exercise 5 matches the slides")

**เขียนคำตอบข้อ 5** (ตอบให้ครบทั้งสามข้อ)

1. สาขา `d=1` มี AICc ต่ำกว่า (`1209.40` เทียบ `1229.36`) แต่ test RMSE แย่กว่า (`2266.97` เทียบ `1655.73`) อธิบายว่าเหตุใด AICc สองค่านี้จึงเปรียบเทียบกันไม่ได้ **ตั้งแต่ต้น** ไม่ใช่เพียง "บังเอิญเลือกผิด"
2. ถ้าอีกกลุ่มหนึ่งรัน grid เดียวกันบนอนุกรมอื่น แล้วพบว่าการเทียบข้าม `d` **เลือกถูก** ผลนั้นหักล้างข้อสรุปของข้อ 1 หรือไม่ เพราะอะไร
3. ถ้าจะเลือกระหว่างสองสาขานี้จริง ๆ ต้องใช้อะไรตัดสิน และเหตุใดการดู test RMSE ในตารางข้างบนแล้วเลือก `d=0` จึงยัง **ไม่ใช่** กระบวนการที่ถูกต้องนัก

_(เขียนคำตอบที่นี่)_

### Residual diagnostics

ตรวจ residual ของ model ที่เลือกไว้ ระวังทิศทางของสมมติฐาน: $H_0$ ของ Ljung-Box คือ "residual เป็น white noise" ดังนั้น **$p$ สูงคือข่าวดี** ซึ่งกลับด้านจาก KPSS

In [ ]:
# ค่า d + D*m แรกไม่ใช่ innovation จริง เป็นช่วงอุ่นเครื่องของ filter
burn = 0 + 1 * 12
resid = pd.Series(read_fit.resid, index=cars_train.index).iloc[burn:]
lb = acorr_ljungbox(resid, lags=[24], return_df=True)
lb_p = float(lb["lb_pvalue"].iloc[0])
print(f"n residuals = {len(resid)}   Ljung-Box lag 24: Q = {float(lb['lb_stat'].iloc[0]):.3f}"
      f", p = {lb_p:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(resid.index, resid.to_numpy())
axes[0].axhline(0, ls=":", color="grey")
axes[0].set_title("Innovation residuals")
rr = acf_by_hand(resid, 24)
axes[1].bar(range(1, 25), rr[1:], width=0.6)
axes[1].axhline(band(len(resid)), ls="--", color="crimson")
axes[1].axhline(-band(len(resid)), ls="--", color="crimson")
axes[1].set_title("ACF of residuals")
plt.tight_layout()
plt.show()

In [ ]:
assert len(resid) == 84
assert abs(float(lb["lb_stat"].iloc[0]) - 33.626) < 0.05
assert abs(lb_p - 0.0915) < 5e-4
assert lb_p > 0.05, "Ljung-Box should not reject white noise here"
print("ok: residual diagnostics match the slides")

## Exercise 6: ETS เทียบกับ ARIMA บน `CM_temp` (16 คะแนน)

ข้อนี้ใช้ **ช่วงทดสอบเดียวกับ Lab 08 เป๊ะ ๆ** คือ train 235 เดือน test 24 เดือน (2017-08 ถึง 2019-07) ตัวเลขของ ETS ที่ได้จึงต้องตรงกับที่ได้ใน Lab 08 และเป็นสิ่งที่ทำให้การเปรียบเทียบนี้ยุติธรรม

ขั้นตอน

1. เลือก `D` และ `d` สำหรับ `temp_m` ด้วยกระบวนการเดียวกับ Exercise 3
2. รัน grid search **ภายใน `d` ที่ตรึงไว้แล้ว**
3. fit ETS `(M,Ad,M)` ซึ่งเป็นผู้ชนะจาก Lab 08
4. วัดทั้งสองกับ seasonal naive บน test set เดียวกัน

In [ ]:
# TODO: เลือก d สำหรับ temp_m โดยใช้ KPSS บนอนุกรมที่ seasonal difference แล้ว
#       เก็บผลการทดสอบไว้ใน temp_tests และค่า d ไว้ใน temp_d
temp_tests = ...
temp_d = ...
print(temp_tests)
print("chosen d =", temp_d)

In [ ]:
assert temp_d == 0, "KPSS does not reject after one seasonal difference"
temp_grid = sarima_grid(temp_train, temp_test, d=temp_d)
best_temp = temp_grid.iloc[0]
print(f"{best_temp['order']}{best_temp['seasonal']}  AICc={best_temp['aicc']:.2f}"
      f"  test RMSE={best_temp['rmse']:.4f}")

In [ ]:
# TODO: fit ETS (M,Ad,M) บน temp_train แล้วคำนวณ test RMSE
#       และคำนวณ RMSE ของ seasonal naive บนช่วงเดียวกัน
#       คำแนะนำ: ETSModel(y, error="mul", trend="add", damped_trend=True,
#                         seasonal="mul", seasonal_periods=12)
ets_fit = ...
ets_rmse = ...
temp_sn = ...

scoreboard = pd.DataFrame([
    {"method": f"SARIMA {best_temp['order']}{best_temp['seasonal']}",
     "rmse": best_temp["rmse"]},
    {"method": "ETS (M,Ad,M), Lab 08's winner", "rmse": ets_rmse},
    {"method": "Seasonal naive (no parameters)", "rmse": temp_sn},
]).set_index("method").sort_values("rmse")
display(scoreboard)

In [ ]:
# ตรงกับตารางบนสไลด์ Lec 09 หน้า "CM_temp: ARIMA ชนะชัดเจน" และกับ Lab 08
assert abs(best_temp["aicc"] - 630.62) < 0.05
assert abs(best_temp["rmse"] - 0.9190) < 5e-4
assert abs(ets_rmse - 1.1060) < 5e-4, "must reproduce Lab 08's number exactly"
assert abs(temp_sn - 1.2227) < 5e-4, "must reproduce Lab 08's number exactly"
assert best_temp["order"] == (0, 0, 2) and best_temp["seasonal"] == (0, 1, 1, 12)
print("ok: Exercise 6 matches the slides and Lab 08")

In [ ]:
print(f"ARIMA beats ETS by            {100 * (1 - best_temp['rmse'] / ets_rmse):.1f}%")
print(f"ARIMA beats seasonal naive by {100 * (1 - best_temp['rmse'] / temp_sn):.1f}%")
print(f"AICc of ARIMA = {best_temp['aicc']:.2f}   AICc of ETS = {ets_fit.aicc:.2f}")

**เขียนคำตอบข้อ 6** (ตอบให้ครบทั้งสี่ข้อ)

1. เรียงอันดับสามวิธีตาม test RMSE และระบุว่า ARIMA ชนะ ETS และชนะ seasonal naive อยู่กี่เปอร์เซ็นต์
2. AICc ของ ARIMA คือ `630.62` และของ ETS คือ `715.97` สรุปได้ไหมว่า ARIMA ดีกว่าเพราะ AICc ต่ำกว่า เพราะอะไร
3. ใน Lab 07 และ Lab 08 benchmark ตามมาติด ๆ หรือชนะเสียด้วยซ้ำ แต่คราวนี้ ARIMA ชนะขาด อะไรใน `CM_temp` ที่ทำให้มันเหมาะกับ ARIMA เป็นพิเศษ
4. บนสไลด์หน้า "elecequip: ARIMA แพ้ และแพ้อย่างน่าสนใจ" `elecequip` ให้ผลตรงข้ามคือ ARIMA แพ้ทั้ง decomposition และ seasonal naive ถ้าได้อนุกรมใหม่มาหนึ่งชุดโดยไม่รู้อะไรเลย ผลสองอันนี้บอกให้ทำอะไรเป็นอย่างแรก

_(เขียนคำตอบที่นี่)_

## โจทย์เสริม (ไม่คิดคะแนน)

1. ลองขยายกล่องการค้นหาเป็น `p, q` $\in \{0,1,2,3\}$ แล้วดูว่า model ที่ AICc เลือกเปลี่ยนไปหรือไม่ และ test RMSE ดีขึ้นหรือแย่ลง
2. `ARIMA(0,1,1)` ควรให้ผลเท่ากับ SES ลองใช้ `temp_m` เฉลี่ยรายปี fit ทั้งสองแบบ แล้ววัดว่าค่าพยากรณ์ห่างกันเท่าไร (บนสไลด์ได้ $0.009$ องศา)
3. ลองใช้ `BIC` แทน `AICc` ในการเลือก แล้วดูว่าได้ model ที่เล็กลงหรือไม่

## Checklist ก่อนส่ง

- [ ] โหลดข้อมูลจาก URL ที่ pin ไว้ และได้ `CarSales` 108 แถว `CM_temp` 7882 แถว `AirPassengers` 144 แถว
- [ ] ทุกอนุกรมเรียก `.asfreq()` แล้ว และ `index.freqstr` ไม่เป็น `None`
- [ ] `cars_train`/`cars_test` = 96/12 และ `temp_train`/`temp_test` = 235/24 โดย test เริ่ม 2017-08
- [ ] `run_tests` อ่านทิศทางของ KPSS และ ADF ถูกทั้งคู่
- [ ] `acf_by_hand` ตรงกับ `statsmodels.acf` ในระดับ $10^{-10}$
- [ ] ตาราง `over` ให้ sd ต่ำสุดที่ `extra_d = 0` และ `r_1` ดิ่งเป็น $-0.539$ เมื่อ difference เกิน
- [ ] `diff_train` มี 84 จุด แถบนัยสำคัญ $\pm 0.214$ และ ACF หลุดที่ lag 1, 2, 12, 17
- [ ] `SARIMA(0,0,2)(0,1,1,12)` ได้ AICc 1229.36 และ test RMSE 1655.7
- [ ] grid ทั้งสองสาขามี 36 fit และสาขา `d=1` มี AICc ต่ำกว่าแต่ RMSE แย่กว่า
- [ ] Ljung-Box ที่ lag 24 ได้ $p = 0.0915$ และตอบได้ว่าเหตุใด $p$ สูงจึงเป็นข่าวดี
- [ ] `CM_temp` ได้ ARIMA 0.9190, ETS 1.1060, seasonal naive 1.2227 ตรงกับ Lab 08
- [ ] ตอบคำถามทุกข้อในทุก markdown cell ที่ขึ้นต้นด้วย "เขียนคำตอบ"
- [ ] restart kernel แล้ว run all ผ่านตลอดโดยไม่มี error
- [ ] เปลี่ยนชื่อไฟล์เป็น `lab09_studentid.ipynb` ก่อนส่ง